# PGM Initial Condition Updater

Wires the per-layer 2D grids of a 3D UZ water-quality (WQ) result into a MIKE SHE `.she`
file as spatial initial conditions.

```
3D UZ result (.dfs3)  --split-->  Layer_1.dfs2 .. Layer_N.dfs2
                                          |
template .she (PFS)  --------------------/--> updated .she
```

## Workflow Overview
1. **Environment**: resolve repo path and import the `initial_condition_updater` module
2. **Configure Inputs**: set the `.dfs3`, the `.she`, output location, timestep, options
3. **Derived Paths**: split directory + output `.she`
4. **Backup**: copy the original `.she` to a timestamped sibling
5. **Split**: write one `Layer_<k>.dfs2` per UZ computational layer (top = `Layer_1`)
6. **Update**: inject per-layer initial conditions and write the new `.she`

Any matching `.she` + `.dfs3` pair can be used, as long as they belong to the same model.

## Step 1: Environment and Module Import

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

REPO_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
SRC_DIR = REPO_ROOT.joinpath("src")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from plant_growth_module import initial_condition_updater as icu  # noqa: E402

print("Repo root:   ", REPO_ROOT)

## Step 2: User-Editable Parameters

Point `DFS3_PATH` and `SHE_INFILE` at a matching result/model pair. `OUTPUT_DIR` is where the
split `Layer_*.dfs2` files and the updated `.she` are written.

- `TIMESTEP`: integer step index (`-1` = last), or an exact timestep, e.g. `"2020-08-31"`.
- `REVERSE_Z`: `True` makes `Layer_1.dfs2` the top of the soil column so `.she [Layer_1]` maps
  directly to `Layer_1.dfs2`. Flip if a MIKE SHE spot-check shows the column is inverted.
- `SPECIES`: `None` updates every matched WQ species; or pass a list of species names.

In [ ]:
# User-editable parameters
MAIN_DIR = Path(
    r"C:\Users\jaan\OneDrive - DHI\_projects\Phishes\Initial_condition_updater"
)

DFS3_PATH = MAIN_DIR.joinpath(
    "raw",
    "Cernici16_Ben_v101_WM_AD_v20_v1_ALMM_JKLparams_macroporeoff_WQ_3DUZ",
    "Cernici16_Ben_v101_WM_AD_v20_v1_ALMM_JKLparams_macroporeoff_WQ_3DUZ.dfs3",
)
SHE_INFILE = MAIN_DIR.joinpath(
    "raw", "Cernici16_Ben_v101_WM_AD_v20_v1_ALMM_JKLparams_Hotstart.she"
)
OUTPUT_DIR = MAIN_DIR.joinpath("processed")

TIMESTEP = -1
REVERSE_Z = True
SPECIES = None  # None = all matched species

print("DFS3:      ", DFS3_PATH)
print("SHE infile:", SHE_INFILE)
print("Output dir:", OUTPUT_DIR)

## Step 3: Derived Paths

In [ ]:
# Split files go next to the output .she so FILE_NAME references stay relative and short.
SPLITTED_DIR = OUTPUT_DIR.joinpath(f"{DFS3_PATH.stem}_splitted")
SHE_OUTFILE = OUTPUT_DIR.joinpath(SHE_INFILE.name)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Split dir:  ", SPLITTED_DIR)
print("SHE outfile:", SHE_OUTFILE)

## Step 4: Backup the Original `.she`

In [ ]:
backup_path = icu.backup_she(SHE_INFILE)

## Step 5: Split the 3D UZ Result into Per-Layer DFS2 Files

Writes `Layer_1.dfs2` .. `Layer_N.dfs2` (one per UZ computational layer). These physical files
are required by the model.

In [ ]:
layer_files = icu.split_dfs3_to_layers(
    DFS3_PATH, SPLITTED_DIR, time=TIMESTEP, reverse_z=REVERSE_Z
)
print(f"Wrote {len(layer_files)} layer files.")

## Step 6: Inject Initial Conditions and Write the Updated `.she`

In [ ]:
summary = icu.update_initial_conditions(
    SHE_INFILE, SHE_OUTFILE, SPLITTED_DIR, species=SPECIES
)

print(f"\nLayers per species: {summary['n_layers']}")
print(f"Species updated:    {summary['n_species_updated']}")
print(f"Output .she:        {summary['outfile']}")
if summary["items_without_species"]:
    print(f"Items without a matching species: {summary['items_without_species']}")